# Phase 3 — Qwen2.5-VL-3B Invoice CSV Fine-tuning (T4)

**Goal:** train an actual 4-bit LoRA adapter on synthetic invoices only; evaluate base versus trained model on synthetic validation. The five Optum test invoices and CORD are deliberately excluded.

**Start:** Runtime → Change runtime type → T4 GPU. Run cells top to bottom. First run only **2 steps** on the included 8 train/2 validation examples; do not claim accuracy gains until measured.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "Select a GPU runtime before continuing"
print("GPU:", torch.cuda.get_device_name(0))

## Upload synthetic-only training bundle

On your **Windows laptop**, run `python -m scripts.prepare_colab_dataset`. Upload the resulting `colab_training_bundle.zip` here. No GitHub repository, CORD, or Optum evaluation invoice needs to be uploaded. Re-running upload cell overwrites current workspace contents.

In [ ]:
from google.colab import files
from pathlib import Path
from zipfile import ZipFile

uploaded = files.upload()
assert "colab_training_bundle.zip" in uploaded, "Upload colab_training_bundle.zip"
workspace = Path("/content/optum_train")
workspace.mkdir(parents=True, exist_ok=True)
with ZipFile("colab_training_bundle.zip") as archive:
    assert not any(name.startswith(("data/invoices/", "data/evaluation/", "data/cord/")) for name in archive.namelist())
    archive.extractall(workspace)
print("Unpacked synthetic-only training bundle:", workspace)
%cd /content/optum_train

## Install Colab dependencies

If Colab asks you to restart runtime, do so and then rerun **from the GPU check and unzip cells**. Install time and model downloads depend on Colab network availability.

In [ ]:
%pip install -q -r requirements-colab.txt

## Validate training and validation splits (CPU)

The training labels are read from paired CSVs. Images and CSVs are checked against `annotations.jsonl` before any GPU model loads.

In [ ]:
from src.vlm.training_data import read_records
train = read_records("data/synthetic", "train")
val = read_records("data/synthetic", "validation")
print("Training images:", len(train), "Validation images:", len(val))
print("First pair:", train[0]["image_path"].name, train[0]["csv_path"].name)
print(train[0]["target_csv"])

## Baseline: evaluate pretrained model BEFORE training

This downloads Qwen2.5-VL-3B and runs two actual validation generations. The report contains both predictions and accuracy numerators/denominators. It may take some time on a free T4.

In [ ]:
!python -m src.vlm.evaluate --dataset-root data/synthetic --split validation --max-samples 2 --output reports/training/baseline.json

## QLoRA training — first 2 optimizer steps

**T4 settings:** 4-bit NF4 base weights, float16 compute, LoRA attention projections (rank 8), microbatch 1, grad accumulation 2, activation checkpointing, bounded vision resolution. The resulting adapter is real only if this cell finishes successfully. If OOM, reduce `--max-pixels` to `200704` (256×28×28) and retry.

In [ ]:
import gc
import torch
gc.collect()
torch.cuda.empty_cache()
!python -m src.vlm.train_lora --dataset-root data/synthetic --max-steps 2 --grad-accum 2 --lora-rank 8 --output-dir models/qwen25-vl-invoice-lora

## Evaluate trained adapter (same two held-out synthetic validation images)

Compare baseline with finetuned metrics; improvement is not guaranteed on only eight training examples.

In [ ]:
!python -m src.vlm.evaluate --dataset-root data/synthetic --split validation --max-samples 2 --adapter models/qwen25-vl-invoice-lora/adapter --output reports/training/finetuned.json

In [ ]:
import json
from pathlib import Path
for tag in ("baseline", "finetuned"):
    p = Path(f"reports/training/{tag}.json")
    report = json.loads(p.read_text())
    print(tag, "field accuracy:", report["field_accuracy_by_row_position"],
          "exact-row recall:", report["exact_row_recall"],
          "exact invoices:", report["invoice_exact_match"])

## Save actual model artifacts and reports to Drive

**Required:** Colab runtime storage disappears. Save both the adapter folder and metrics. If Drive quota is tight, do not copy Hugging Face base-model cache; it can be downloaded again.

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil
drive.mount("/content/drive")
destination = Path("/content/drive/MyDrive/optum_phase3")
destination.mkdir(parents=True, exist_ok=True)
shutil.copytree("models/qwen25-vl-invoice-lora/adapter", destination / "adapter", dirs_exist_ok=True)
shutil.copytree("reports/training", destination / "reports", dirs_exist_ok=True)
for filename in ("run_config.json", "training_result.json", "training_log.json"):
    shutil.copy2(Path("models/qwen25-vl-invoice-lora") / filename, destination / filename)
print("Saved:", destination)

## Later (not part of the first smoke test)

On the laptop, generate a larger synthetic dataset (100–300 images) **after** confirming this notebook works. Rebuild `colab_training_bundle.zip`, upload again, and increase optimizer steps based on GPU memory and actual validation scores. **Do not start DPO or Optum final testing yet.**